In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import torch
import csv
import optuna
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast
from transformers import BertForTokenClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [ ]:
# Load the expanded datasets
expanded_train_data = pd.read_csv('final data/expanded_train_data.csv')
expanded_validation_data = pd.read_csv('final data/expanded_validation_data.csv')
expanded_test_data = pd.read_csv('final data/expanded_test_data.csv')

In [ ]:
# Prepare the dataset for training

class SPODataset(Dataset):
    """
    Manages and prepares the data for training BERT for SPO token-level labeling. 
    
    """
    def __init__(self, dataframe, tokenizer, max_len=512, stride=300):
        mask = dataframe['spo_label'].apply(lambda x: x != 'other').groupby(dataframe['sentence_id']).transform('any')  # Filter out sentences where all tokens are labeled with 'other'
        self.data = dataframe[mask]
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.label_dict = self.create_label_dict()
        self.items = self.create_items()


    def create_label_dict(self):
        """
        Created a dictionary that map the SPO-labels (textual labels) to numeric indices.

        """
        unique_labels = ['Subject', 'Predicate', 'Object', 'other']

        return {label: idx for idx, label in enumerate(unique_labels)}

    def align_labels_with_tokens(self, original_tokens, labels):
        """
        Aligns the SPO-labels with their respective tokens after tokenization.
        
        """
        tokenized_input = self.tokenizer(original_tokens, is_split_into_words=True, return_offsets_mapping=True, padding='max_length', truncation=False, max_length=self.max_len)
        aligned_labels = []
        previous_word_idx = None  # Track the index of the last non-subtoken word

        for i, token in enumerate(tokenized_input.tokens()):
            word_idx = tokenized_input.word_ids()[i]

            # Assign actual label to the first subtoken and dummy label to subsequent subtokens
            if word_idx is not None:
                if word_idx != previous_word_idx:  # Check if it's the start of a new word
                    aligned_label = self.label_dict[labels[word_idx]]
                    previous_word_idx = word_idx
                else:
                    aligned_label = -100  # Assign a dummy label (-100) for subtokens following the first one
            else:
                aligned_label = -100  # Assign the label -100 for special tokens like [CLS], [SEP], [PAD], ignoring them during the training

            aligned_labels.append(aligned_label)
            # print(f"{token}\t\t{aligned_label}")  # Print the token with its numeric aligned label

        return tokenized_input, aligned_labels
    
    def create_items(self):
        """
        Prepares tokenized data into manageable pieces (sliding windows) for training.

        """
        items = []
        all_tokens = []
        all_labels = []
        # all_sentence_ids = []

        # Flatten all tokens and labels into single lists
        for _, row in self.data.iterrows():
            all_tokens.append(row['token'])  
            all_labels.append(row['spo_label']) 
            # all_sentence_ids.append(row['sentence_id']) 

        # Tokenize the entire sequence of tokens and align both labels and sentence IDs
        # tokenized_input, aligned_labels, aligned_sentence_ids = self.align_labels_and_sentence_ids_with_tokens(all_tokens, all_labels, all_sentence_ids)
        tokenized_input, aligned_labels = self.align_labels_with_tokens(all_tokens, all_labels)
        
        # Ensure sliding window covers the entire sequence
        total_tokens = len(tokenized_input['input_ids'])
        print('The total tokens are:', total_tokens)

        # Apply sliding window technique
        start_index = 0

        while start_index < total_tokens:
            # Define the end index for the window
            end_index = min(start_index + self.max_len, total_tokens)
            
            # Extract the token sequence and label sequence for this window
            segment_input_ids = tokenized_input['input_ids'][start_index:end_index]
            segment_attention_mask = tokenized_input['attention_mask'][start_index:end_index]
            segment_labels = aligned_labels[start_index:end_index]
            # segment_sentence_ids = aligned_sentence_ids[start_index:end_index]

            # Print the tokens and labels for this window
            print(f"Tokens for window starting at {start_index}: {self.tokenizer.convert_ids_to_tokens(segment_input_ids)}")
            print(f"Labels for window starting at {start_index}: {segment_labels}")
            print("=" * 80)  # Separator for better readability
            
            # Append the sequence to the items list
            items.append({
                'input_ids': torch.tensor(segment_input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(segment_attention_mask, dtype=torch.long),
                'labels': torch.tensor(segment_labels, dtype=torch.long)
                # 'sentence_ids': torch.tensor(segment_sentence_ids, dtype=torch.long)
            })

            # Move the start index forward by the stride length
            start_index += self.stride

        return items

    def __getitem__(self, index):
        return self.items[index]


    def __len__(self):
        return len(self.items) # The length of items represents the total sliding windows created


# Initialize tokenizer
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Initialize dataset
training = SPODataset(expanded_train_data, tokenizer)
validation = SPODataset(expanded_validation_data, tokenizer)
testing = SPODataset(expanded_test_data, tokenizer)

# Print the total number of sliding window items for each dataset
print(f"Training dataset initialized with {len(training)} items.")
print(f"Validation dataset initialized with {len(validation)} items.")
print(f"Testing dataset initialized with {len(testing)} items.")

# Print the label dictionary
print("Label Dictionary Training Data:", training.label_dict)
print("Label Dictionary Validation Data:", validation.label_dict)
print("Label Dictionary Testing Data:", testing.label_dict)

The total tokens are: 42063
Tokens for window starting at 0: ['[CLS]', 'i', 'have', 'been', 'feeling', 'quite', 'thirsty', 'lately', 'and', 'drinking', 'much', 'more', 'water', 'than', 'usual', '.', 'i', 'have', 'been', 'feeling', 'quite', 'thirsty', 'lately', 'and', 'drinking', 'much', 'more', 'water', 'than', 'usual', '.', 'is', 'this', 'linked', 'to', 'my', 'diabetes', '?', 'unusually', 'high', 'thirst', 'can', 'be', 'a', 'sy', '##mpt', '##om', 'of', 'higher', 'than', 'normal', 'blood', 'sugar', 'levels', '.', 'when', 'there', 'is', 'excess', 'sugar', 'in', 'your', 'blood', ',', 'your', 'kidney', '##s', 'work', 'harder', 'to', 'filter', 'and', 'absorb', 'it', ',', 'creating', 'a', 'need', 'for', 'more', 'fluids', 'in', 'your', 'body', '.', 'and', 'is', 'this', 'excess', 'thirst', 'harmful', '?', 'a', ':', 'if', 'left', 'un', '##che', '##cked', ',', 'high', 'blood', 'sugar', 'levels', 'can', 'lead', 'to', 'complications', 'like', 'de', '##hy', '##dra', '##tion', ',', 'which', 'can', 

In [ ]:
# Training and Hyperparameter Tuning

def collate_fn(batch):
    """
    Ensures that every input sequence fed into the model during training or inference is consistent in length.
    
    """
    input_ids = pad_sequence([item['input_ids'] for item in batch], batch_first=True, padding_value=0)
    attention_mask = pad_sequence([item['attention_mask'] for item in batch], batch_first=True, padding_value=0)
    labels = pad_sequence([item['labels'] for item in batch], batch_first=True, padding_value=-100)
    
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}


def train_model(model, train_loader, device, optimizer, scheduler):
    """
    Trains the BERT model for one epoch.
    
    """
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Reset gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

        # Calculate loss
        loss = outputs.loss
        
        # Backward pass
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 

        # Update weights
        optimizer.step()

        # Adjusting the learning rate using a scheduler
        scheduler.step()
        
        # Accumulate loss
        total_loss += loss.item()

        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)
        preds = preds.view(-1).cpu().numpy()
        labels = labels.view(-1).cpu().numpy()
        mask = labels != -100
        all_preds.extend(preds[mask])
        all_labels.extend(labels[mask])

    train_f1 = f1_score(all_labels, all_preds, average='macro')
    avg_loss = total_loss / len(train_loader)

    return avg_loss, train_f1


def evaluate_model(model, val_loader, device):
    """
    Evaluates the BERT model on the validation dataset.
    
    """
    # Set the model to evaluation mode
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)
            
            preds = preds.view(-1).cpu().numpy()
            labels = labels.view(-1).cpu().numpy()
            # mask = labels != -100     # Exclude the padding tokens for the f1-score calculation
            mask = (labels != -100) & (labels != 3)  # Exclude 'other' and padding tokens for the f1-score calculation
            all_preds.extend(preds[mask])
            all_labels.extend(labels[mask])

    return f1_score(all_labels, all_preds, average='macro')


def train_and_evaluate_with_early_stopping(model, train_loader, val_loader, device, optimizer, scheduler, patience=2):
    """
    Trains the model with early stopping based on validation f1-score.
    
    """

    # 'patience' is the number of epochs to continue training without improvement in the f1-score before stopping. 

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(10):  # Set a high maximum number of epochs
        avg_train_loss, train_f1 = train_model(model, train_loader, device, optimizer, scheduler)
        val_f1 = evaluate_model(model, val_loader, device)
        
        print(f"Epoch {epoch + 1} - Train Loss: {avg_train_loss:.4f}, Train F1: {train_f1:.4f}, Val F1: {val_f1:.4f}")
        
        # Early stopping logic based on validation f1-score
        if val_f1  > best_f1:
            best_f1 = val_f1 
            patience_counter = 0
            print(f"New best F1: {best_f1:.4f}, saving model...")
            torch.save(model, "best_model_spo.pth")  # Saving the entire model
        else:
            patience_counter += 1
            print(f"No improvement, patience counter: {patience_counter}")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break

    model = torch.load("best_model_spo.pth")  # Load the entire model

    return best_f1


def main(trial):
    """
    Main function to set up data loaders, model, optimizer, and scheduler, and run training with hyperparameter tuning.
    
    """
    # Suggest hyperparameters for optimization
    learning_rate = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [1, 8, 16])
    
    # Set up data loaders
    train_loader = DataLoader(training, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(validation, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # Initialize model, optimizer, and scheduler
    model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=len(training.label_dict))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    
    num_training_steps = len(train_loader) * 3
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

    try:
        best_f1 = train_and_evaluate_with_early_stopping(model, train_loader, val_loader, device, optimizer, scheduler)
        print(f"Returning F1 Score: {best_f1}")
        return best_f1
    except Exception as e:
        print(f"An error occurred in main: {e}")
        return 0  # Provide a default F1 score in case of error


def objective(trial):
    """ 
    Objective function for Optuna hyperparameter optimization.
    
    """
    return main(trial)

# Hyperparameter optimization using Optuna
study = optuna.create_study(direction='maximize')  # 'maximize': find hyperparameters that yield the highest value of the objective function
study.optimize(objective, n_trials=10)  

In [ ]:
# Evaluation on the Test Set

# Load the best model
best_model_path = 'cte/bert/best_model.pth'
model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=len(training.label_dict))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)

# Prepare the DataLoader for the test set
test_loader = DataLoader(testing, batch_size=1, shuffle=False, collate_fn=collate_fn)


def evaluate_test_set(model, test_loader, device):
    """
    Compute metrics and create the confusion matrix.
    
    """
    model.eval()  # Set the model to evaluation mode
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)

            preds = preds.view(-1).cpu().numpy()
            labels = labels.view(-1).cpu().numpy()
            mask = labels != -100  # Exclude padding tokens
            all_preds.extend(preds[mask])
            all_labels.extend(labels[mask])

    # Compute the confusion matrix
    cm = confusion_matrix(all_labels, all_preds, labels=list(training.label_dict.values()))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(training.label_dict.keys()))

    # Compute other metrics
    precision = precision_score(all_labels, all_preds, average='macro')
    recall = recall_score(all_labels, all_preds, average='macro')
    f1 = f1_score(all_labels, all_preds, average='macro')

    # Display the confusion matrix
    disp.plot(cmap=plt.cm.Blues)
    plt.show()

    return precision, recall, f1


# Evaluate and print metrics
precision, recall, f1 = evaluate_test_set(model, test_loader, device)
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
